## 1. Motivation and model overview

The **ball-drive risk** component estimates the probability that the current ball carrier loses possession during a canonical one-second ball drive. It is the action-specific risk $d_t$ used by PRESS to construct the possession-loss hazard. The event concerns loss of the current player's control during the drive, not a failed pass.

The model combines two complementary research traditions. Bekkers' pressing-intensity formulation motivates estimating how quickly defenders can reach a moving ball carrier [@bekkers2025]. Spearman's pitch-control framework motivates representing arrival uncertainty and the rate at which a player can establish control [@spearman2017; @spearman2018beyond]. PRESS adapts these ideas to a player **already in possession**: a defender must be able to reach the carrier and, once reachable, can generate a successful defensive intervention at a certain *tackle rate*. This rate plays a role analogous to the control rate in pitch control, but measures the intensity of possession loss. PRESS does not reproduce Bekkers' aggregate pressing score.

Each defender contributes a time-varying cause-specific hazard. A shared survival process combines these contributions into the probability of retaining the ball over one second. The one-second action convention follows the modular EPV perspective of Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021].

The approach is **physics-informed**: motion and reachability follow an explicit physical approximation, whereas tackle intensity is estimated from observed outcomes. The tackle rate is represented by a penalized additive spline model fitted to observed outcomes.

## 2. Motion and defender interception time

### Projected ball-carrier trajectory

Let $x_B(0)$ and $v_B(0)$ denote the carrier's observed position and velocity at the evaluation frame. During the hypothetical drive, the carrier follows a constant-velocity projection:

$$
x_B(t)=x_B(0)+v_B(0)t,\qquad 0\le t\le 1.
$$

At each integration time, defender $i$ targets the projected carrier position $x_B(t)$. The carrier is represented as a 2d point. Let $d_i(t)$ denote the defender's distance to that target, and let $u_i=\lVert v_i(0)\rVert$ be the defender's observed initial speed.

### Travel and turning time

For initial speed $u_i\le v_{\max}$, the motion approximation uses linear acceleration $a=a_{\mathrm{linear}}$ up to an effective speed $v_{\max}$. Define

$$
t_a=\max\left\{\frac{v_{\max}-u_i}{a},0\right\},
\qquad
d_a=u_it_a+\frac12 at_a^2.
$$

The estimated travel time to the projected target is

$$
t_{\mathrm{move},i}(t)=
\begin{cases}
\dfrac{\sqrt{u_i^2+2ad_i(t)}-u_i}{a}, & d_i(t)\le d_a,\\[6pt]
t_a+\dfrac{d_i(t)-d_a}{v_{\max}}, & d_i(t)>d_a.
\end{cases}
$$

If the observed initial speed exceeds $v_{\max}$, the model retains that initial speed without additional acceleration and uses $t_{\mathrm{move},i}(t)=d_i(t)/u_i$. Thus $v_{\max}$ is not a hard cap on observed initial speeds.

Following Pleuler's use of a heading-dependent turning penalty [@pleuler2020handbook], PRESS parameterizes the defender's reorientation with a simplified turning delay. With turning angle $\theta_i(t)$ in radians and turning-acceleration parameter $a_{\mathrm{turn}}$,

$$
t_{\mathrm{turn},i}(t)=\frac{u_i\theta_i(t)}{a_{\mathrm{turn}}}.
$$

This expression follows from a turning radius $R=u_i^2/a_{\mathrm{turn}}$ and arc time $R\theta_i/u_i$. The implementation adds a delay but does not propagate the defender along a turning arc. The arrival-time estimate is

$$
\tau_i(t)=t_{\mathrm{move},i}(t)+t_{\mathrm{turn},i}(t).
$$

For the existing near-target convention, $\tau_i(t)=0$ whenever $d_i(t)\le0.1$ m. This does not imply certain reachability.

## 3. Probabilistic defender reachability

As in the pitch-control approach, deterministic arrival times are converted into a smooth reachability gate. With logistic arrival-time dispersion $s$ (in seconds), defender $i$ has reachability

$$
F_i(t)=\left[1+\exp\left\{-\frac{\pi\,[t-\tau_i(t)]}{\sqrt{3}\,s}\right\}\right]^{-1}.
$$

The parameter $s$ is the standard deviation of the corresponding logistic arrival-time uncertainty. The gate increases when the defender can reach the projected carrier position sooner relative to elapsed time.

## 4. Tackle intensity

### Additive spline tackle rate

The second ingredient is the instantaneous rate at which a reachable defender can cause loss of control. Inspired by the control-rate mechanism of pitch control, PRESS assigns defender $i$ a nonnegative tackle rate $\lambda_i(t)$, with units s$^{-1}$:

$$
\boxed{\log\lambda_i(t)=\alpha+g_v(\lVert v_B\rVert)+g_B(B(t))+g_\phi(\phi_i(t)).}
$$

Here $\alpha=\log\lambda_0$ is the log baseline tackle-rate parameter. The fitted specification uses linear terms plus restricted cubic nonlinear terms for speed and boundary distance; angle has a standardized linear term and two nonlinear terms. There are no interaction terms. The cause-specific hazards vary with time through defender reachability and the evaluated covariates.

### Why these covariates?

**Carrier speed.** Schepers et al. identify movement speed as an important predictor of dribbling outcomes [@schepers2025dribble]. Greater speed can make fine ball control more difficult, increasing vulnerability to intervention. The fitted PRESS model supports an increase in tackle intensity at higher carrier speeds. The spline allows the magnitude and shape of that effect to be estimated without imposing a strictly log-linear response.

**Boundary proximity.** Available space is a central aspect of defensive pressure, including SkillCorner's emphasis on spatial constraints [@skillcorner2026]. Near a boundary, the carrier has fewer possible directions in which to evade an intervention. PRESS uses the signed minimum distance to **all four** pitch boundaries, recomputed along the projected path:

$$
B(t)=\min\!\left\{\frac L2-x_B(t),\frac L2+x_B(t),\frac W2-y_B(t),\frac W2+y_B(t)\right\}.
$$

This differs from a sideline-only distance and can become negative beyond the pitch boundary.

**Defender approach angle.** Spatial constraints also depend on whether a defender approaches from ahead, behind, or the side. Motivated by the role of the carrier's available field of view and escape directions in pressure interpretation, PRESS measures $\phi_i(t)$ as the angle between carrier velocity and the defender's predicted final approach direction towards $x_B(t)$. It is a geometric proxy, not observed body orientation or gaze. The angle contribution is masked when carrier speed is below 1 m/s or the direction is undefined.

These three covariates describe complementary mechanisms: control difficulty at speed, geometric restrictions near the pitch boundary, and the direction of defensive engagement. The spline terms quantify their associations in the fitted model.


### Spline basis and regularization

The speed and boundary effects are treated as the primary covariates; each combines a linear term with three restricted-cubic nonlinear terms, using five knots:

$$g_v(v)=\beta_v v+\sum_{k=1}^{3}\gamma_{vk}b_{vk}(v),\qquad
 g_B(B)=\beta_B B+\sum_{k=1}^{3}\gamma_{Bk}b_{Bk}(B).$$

The defender approach angle is included as an additional, more parsimoniously parameterized effect, using four knots, with a standardized linear term and two nonlinear terms:

$$g_\phi(\phi)=\beta_\phi z_\phi(\phi)+\sum_{k=1}^{2}\gamma_{\phi k}b_{\phi k}(\phi).$$

The nonlinear terms are regularized by penalizing spline curvature. For each covariate $j\in\{v,B,\phi\}$, let $\boldsymbol\gamma_j$ denote its nonlinear coefficients and $\mathbf b_j(x)$ its nonlinear basis. The quadratic curvature penalty is

$$\frac{\rho_j}{2}\boldsymbol\gamma_j^\top\Omega_j\boldsymbol\gamma_j,$$

where $\Omega_j$ is the curvature penalty matrix associated with the basis and $\rho_j$ controls the strength of regularization. This discourages unnecessary curvature without constraining the direction of the fitted effect. The corresponding Gaussian prior is specified in Section 7; the selection of $\rho_j$ through cross-validation is documented in the [benchmark](../../benchmarks/01_ball_drive_risk.qmd).

## 5. From defender hazards to possession-loss probability

For each defender $i\in\mathcal D$, let $P_i(t)$ be the probability that defender $i$ has caused the **first possession loss** by time $t$. Define $S(t)$ as the probability that the carrier has retained possession up to time $t$. Since the possible first-loss causes compete for the same possession,

$$
S(t)=1-\sum_{j\in\mathcal D}P_j(t).
$$

The model assigns defender $i$ the following differential equation:

$$
\boxed{
\frac{dP_i(t)}{dt}
=\underbrace{\left[1-\sum_{j\in\mathcal D}P_j(t)\right]}_{S(t)}
\underbrace{\lambda_{\mathrm{tackle}}\!\left(\mathbf x_B(t),\mathbf v_B(t),\mathbf x_i(0),\mathbf v_i(0)\right)}_{\lambda_i(t)}
\underbrace{F_i\!\left(\mathbf x_B(t),t\right)}_{F_i(t)}.
}
$$

The initial conditions are $P_i(0)=0$ for every defender and $S(0)=1$. The reachability term $F_i(\mathbf x_B(t),t)$ is the logistic gate defined above, evaluated for the **moving projected carrier position**. The tackle intensity is the fitted nonnegative rate described in Section 4; its compact state-dependent notation here is shorthand for the specified speed, boundary-distance and approach-angle covariates, not an additional fitted dependence on the full position or velocity vectors. The defender state $\mathbf x_i(0),\mathbf v_i(0)$ is observed at the evaluation frame, rather than propagated as a simulated defensive trajectory.

The factor $S(t)$ makes the causes mutually exclusive: once possession has been lost, no other defender can produce its first loss. Thus a high tackle intensity contributes little when the defender is unlikely to reach the carrier, and even a reachable defender contributes only while possession survives.

Equivalently, defining the cause-specific hazard $h_i(t)=\lambda_i(t)F_i(t)$ gives

$$
\frac{dS(t)}{dt}=-S(t)\sum_{i\in\mathcal D}h_i(t),
\qquad
S(t)=\exp\!\left[-\int_0^t\sum_{i\in\mathcal D}h_i(u)\,du\right].
$$

The corresponding integral form is

$$
P_i(t)=\int_0^t S(u)h_i(u)\,du,
\qquad
\boxed{P(\mathrm{loss\ by}\ t\mid\mathrm{drive})=\sum_{i\in\mathcal D}P_i(t)=1-S(t).}
$$

The **one-second ball-drive risk** passed to the wider PRESS framework is $d_t=1-S(1)$, where the evaluation state is understood to be fixed at the start of the hypothetical action. Multiple defenders affect risk through their individual reachability and rates; coordinated defensive behaviour is not explicitly modelled.

## 6. Motion parameters

The model uses $v_{\max}=7$ m/s and $a_{\mathrm{linear}}=a_{\mathrm{turn}}=7$ m/s$^2$. These are effective common movement capabilities, motivated by published models and checked against the tracking distribution.

| Parameter | Literature rationale | Tracking consistency check |
|---|---|---|
| Defender speed $v_{\max}=7$ m/s | Approximately the 25.1 km/h sprint threshold in Bradley et al. [@bradley2009running]. | Defensive-engagement event frames with speed $>2$ m/s: Q95 = **7.050 m/s**. |
| Linear acceleration $a_{\mathrm{linear}}=7$ m/s$^2$ | The acceleration used by Spearman's movement model [@spearman2018beyond]. | Engagement acceleration peaks above 3 m/s$^2$: Q99 = **6.966 m/s$^2$**. |
| Turning acceleration $a_{\mathrm{turn}}=7$ m/s$^2$ | A heading-dependent delay follows the idea in Pleuler [@pleuler2020handbook]; PRESS sets its scale equal to the linear-acceleration parameter. | Equality with linear acceleration is a modelling convention; it is not separately identified by these quantiles. |

The speed filter removes stationary/walking observations for this descriptive check. These conditional quantiles are consistency checks for rounded model scales, not physiological ceilings or independently estimated caps. Event-frame observations and event peaks use different sampling units. The [speed audit](../../data/examples/ball_drive/defender_speed_check.csv), [engagement quantiles](../../data/benchmarks/ball_drive/engagement_quantiles.csv) and [kinematic conventions](../../docs/kinematic-parameter-conventions.md) provide the definitions and sources.

Positions and velocities enter the model directly; observed acceleration is not an input. The current velocities use the offline centred seven-frame tracking estimate.


## 7. Likelihood, priors and parameter estimation 

### Likelihood: the outcome of a one-second carry

For carry $j$, let $Y_j=1$ denote a loss and $Y_j=0$ successful retention over the canonical horizon. Given the observed initial state and parameters $\theta$, the competing-risk model supplies

$$d_j(\theta)=1-S_j(1;\theta).$$

The probability $d_j(\theta)$ depends on the spline coefficients through the defender-specific tackle rates. In particular, for defender $i$ in carry $j$,

$$
\lambda_{ij}(t;\theta)=\exp\!\left\{\alpha+g_v(\lVert v_{Bj}\rVert)+g_B(B_j(t))+g_\phi(\phi_{ij}(t))\right\},
$$

where the $g$ functions are the restricted cubic spline effects defined in Section 4. Combined with the reachability gate, these rates determine the survival probability:

$$
S_j(1;\theta)=\exp\!\left[-\int_0^1\sum_{i\in\mathcal D_j}\lambda_{ij}(t;\theta)F_{ij}(t;s)\,dt\right].
$$

The outcome likelihood is Bernoulli:

$$\boxed{\mathcal L(\theta;D)=\prod_j
 d_j(\theta)^{Y_j}\,[1-d_j(\theta)]^{1-Y_j}.}$$

Equivalently,

$$\log\mathcal L=\sum_j\left\{Y_j\log[1-S_j(1)]+(1-Y_j)\log S_j(1)\right\}.$$

A loss before one second contributes $1-S_j(1)$: its observed failure time does not shorten the forecast horizon. A successful full second contributes $S_j(1)$. The fit learns which initial configurations make the one-second outcome more or less likely; it does not fit the exact tackle time or the observed tackler's identity.

### Priors: positive scales and regularized covariate effects

Write $\alpha=\log\lambda_0$ and $\ell=\log s$. The fitted model uses independent priors

$$\alpha\sim\mathcal N(0,1),\qquad
\beta_v,\beta_B\sim\mathcal N(0,1),$$

$$\boxed{s\sim\operatorname{Gamma}(\text{shape}=2.5,\ \text{rate}=5).}$$

Thus $\lambda_0$ has a lognormal prior with log-location 0 and log-SD 1. The Gamma prior keeps $s$ positive, has mean **0.5 s** and SD **0.316 s**, and places it on the same broad time scale as the arrival dispersion in Spearman's models [@spearman2017; @spearman2018beyond]. These exact prior choices belong to PRESS; they are not published estimates copied from those papers.

The Gaussian priors on the linear speed and boundary effects were informed by preliminary estimates from a log-linear model,

$$\log\lambda(t)=\alpha+\beta_v\lVert v_B\rVert+\beta_B B(t),$$

which yielded coefficient estimates compatible with a standard normal prior. The angle effect was introduced subsequently, with its linear term standardized to accommodate the same prior, $\beta_\phi\sim\mathcal N(0,1)$.

For each nonlinear spline coefficient vector $\boldsymbol\gamma_j$, the curvature penalty in Section 4 corresponds to a zero-centered Gaussian prior with precision $\rho_j\Omega_j$:

$$p(\boldsymbol\gamma_j\mid\rho_j)\propto\exp\!\left(-\frac{\rho_j}{2}\boldsymbol\gamma_j^\top\Omega_j\boldsymbol\gamma_j\right).$$

When $\Omega_j$ is positive definite, this is $\boldsymbol\gamma_j\sim\mathcal N(\mathbf0,(\rho_j\Omega_j)^{-1})$. The curvature matrices encode the regularization of the nonlinear terms; the multipliers $\rho_j$ are selected through cross-validation, as detailed in the benchmark.

The posterior combines the one-second likelihood with these priors:

$$p(\theta\mid D)\propto\mathcal L(\theta;D)\,p(\theta).$$

Optimization uses the log coordinates. Consequently, the prior density for $\ell$ includes the transformation Jacobian:

$$p_\ell(\ell)=p_s(e^\ell)e^\ell
\ \propto\ \exp\{2.5\ell-5e^\ell\}.$$

With the smoothing multipliers held fixed, the minimized negative log posterior, up to constants, is

$$
-\log\mathcal L(\theta;D)
+\frac12(\alpha^2+\beta_v^2+\beta_B^2+\beta_\phi^2)
+5e^\ell-2.5\ell
+\frac12\sum_{j\in\{v,B,\phi\}}\rho_j\boldsymbol\gamma_j^\top\Omega_j\boldsymbol\gamma_j.
$$

Thus the linear angle coefficient has its own standard Gaussian penalty, while curvature regularization acts only on the nonlinear spline coefficients.

### MAP estimation and interpretation

The full parameter vector, including the baseline tackle rate, arrival dispersion and spline coefficients, is estimated jointly by maximum a posteriori (MAP) optimization of the objective above. The selected regularization strengths are treated as fixed for this fit. The resulting point estimates parameterize the model distributed with PRESS; predictions are not averaged over posterior draws.

The joint MAP fit on the complete development cohort gives an arrival-time dispersion of $s=0.299632$ s and a baseline tackle rate of $\lambda_0=0.344563$ s$^{-1}$. These are fitted parameters of the deployed model, not cross-validated performance estimates. The conditional spline curves below illustrate how the fitted tackle rate varies with the three covariates before multiplication by defender reachability.

### Fitted spline curves

![Conditional tackle-rate curves from the bundled MAP fit. Each panel varies one covariate, holding the others at the values shown above it. Rates are shown before multiplication by defender reachability. The speed panel starts at 1 m/s, where the angle term is active.](../../figures/vignette_01_tackle_splines.png)

The dispersion parameter is model-specific. Spearman's pass model reports 0.45 s, while a later OBSO formulation reports a MAP estimate of 0.54 s under a different estimation setup [@spearman2017; @spearman2018beyond]. These values provide context for the magnitude of $s$, not a like-for-like comparison: differences in motion assumptions, cohort construction and tackle-rate flexibility may all contribute, and their individual effects have not been isolated. The [pass-probability vignette](../02_pass_probability/02_pass_probability.ipynb) discusses its independently fitted arrival parameter.

The [ball-drive benchmark](../../benchmarks/01_ball_drive_risk.qmd) documents regularization selection, posterior uncertainty, cross-validation results and comparisons of predictive performance.


## 8. Real-match examples

Set `SKILLCORNER_OPENDATA` to your local open-data repository. [Data setup](../../data.qmd).

### Melbourne Victory–Auckland


In [7]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
from PRESS import PRESS
from PRESS.model import BallDriveModel
from PRESS.visualization import plot_pi

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/processed/2026-10-15_training_v5_outfield/matches/2017461"
press = PRESS.from_skillcorner(match_dir)

frame_id = 6955
state = press.get_state(frame_id)
press.compute_ball_drive_loss(frame_id, model=BallDriveModel.pretrained())
plot_pi(press, frame_id)
plt.savefig("figures/vignette_01_drive_pitch.png", dpi=170)
plt.savefig("figures/vignette_01_drive_pitch.svg")
plt.close()


![Initial frame. Colors show each defender’s share of total loss risk; the title reports the absolute probability.](../../figures/vignette_01_drive_pitch.png)


In [9]:
from PRESS.visualization import plot_frame

end_frame = 6967
plot_frame(press, end_frame)
plt.savefig("figures/vignette_01_drive_end_pitch.png", dpi=170)
plt.savefig("figures/vignette_01_drive_end_pitch.svg")
plt.close()


![Final frame, using `PRESS.visualization.plot_frame`.](../../figures/vignette_01_drive_end_pitch.png)


### Auckland–Newcastle

In [12]:
match_dir = opendata / "data/processed/2026-10-15_training_v5_outfield/matches/1886347"
press = PRESS.from_skillcorner(match_dir)

frame_id = 32937
state = press.get_state(frame_id)
press.compute_ball_drive_loss(frame_id, model=BallDriveModel.pretrained())
plot_pi(press, frame_id)
plt.savefig("figures/vignette_01_drive_own_half_pitch.png", dpi=170)
plt.savefig("figures/vignette_01_drive_own_half_pitch.svg")
plt.close()


![Initial frame. Colors show each defender’s share of total loss risk; the title reports the absolute probability.](../../figures/vignette_01_drive_own_half_pitch.png)

In [14]:
end_frame = 32953
plot_frame(press, end_frame)
plt.savefig("figures/vignette_01_drive_own_half_end_pitch.png", dpi=170)
plt.savefig("figures/vignette_01_drive_own_half_end_pitch.svg")
plt.close()


![Later frame, using `PRESS.visualization.plot_frame`.](../../figures/vignette_01_drive_own_half_end_pitch.png)

## Scope and limitations

The model uses a point-carrier approximation, constant carrier velocity over the hypothetical drive, simplified defender turning, and a reachability-gated tackle process. It does not explicitly represent body posture, foot-level ball control, individual skill, or fully coordinated defensive behaviour. Schepers et al. [@schepers2025dribble] motivate investigating richer dribbling and posture features, but overlap between successful and unsuccessful examples in the current 2D covariates does not establish that all useful 2D information has been exhausted. Shorter-horizon probabilities follow from the survival formulation but have not been separately calibrated.

## References and next steps

Bekkers [@bekkers2025] motivates moving-target pressing and reachability. Spearman [@spearman2017; @spearman2018beyond] motivates arrival uncertainty and control rates. Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021] motivate the canonical action decomposition. Stalpaert et al. [@stalpaert2022] provide a carry-model benchmark, and Schepers et al. [@schepers2025dribble] motivate the dribbling covariates. SkillCorner [@skillcorner2026] motivates the interpretation of spatial constraints.

The equations above describe the PRESS formulation rather than a reproduction of any single cited model. Cohort selection, held-out results and comparisons are documented in the [ball-drive benchmark](../../benchmarks/01_ball_drive_risk.qmd). The pass-success component is described in the [next model vignette](../02_pass_probability/02_pass_probability.ipynb).